# Compare `needs_review` with ISSP final

Run the cells from top to bottom. This notebook takes rows whose `status` is `needs_review` from `outputV2.csv`, compares them with `ISSP Bibliography_final.ris`, prints the counts, and writes a new CSV containing only the rows that still need review. It does not change either input file.

**Exact link rule:** the normalized title must match, and at least one candidate link must match a final-file link. DOI strings and `doi.org` links are treated as the same DOI. Other URLs must match exactly after trimming surrounding whitespace; the notebook does not treat a similar title, redirected URL, or merely shared domain as an exact match. A link that matches a different title stays in the review CSV for safety.

In [1]:
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path
from urllib.parse import unquote, urlsplit
import re
import unicodedata

import pandas as pd

# Change this folder only if the files have moved.
DATA_DIR = Path(r'D:\UoA\Master\2026 S2\Project\DataSet')
OUTPUT_V2 = DATA_DIR / 'outputV2.csv'
FINAL_RIS = DATA_DIR / 'ISSP Bibliography_final.ris'
REVIEW_CSV = DATA_DIR / 'needs_review_after_final.csv'

for source_file in (OUTPUT_V2, FINAL_RIS):
    if not source_file.is_file():
        raise FileNotFoundError(f'Cannot find input file: {source_file}')
print('Input files found. The comparison has not run yet.')

Input files found. The comparison has not run yet.


In [2]:
RIS_TAG = re.compile(r'^\s*([A-Za-z0-9]{2})\s*-\s*(.*?)\s*$')
DOI_RE = re.compile(r'^10\.\d{4,9}/\S+$', re.IGNORECASE)

def clean(value):
    return str(value or '').strip()

def title_key(value):
    text = unicodedata.normalize('NFKC', clean(value))
    return re.sub(r'\s+', ' ', text).casefold()

def link_key(value):
    text = clean(value)
    if not text:
        return ''
    parsed = urlsplit(text)
    if parsed.hostname and parsed.hostname.casefold() in {'doi.org', 'dx.doi.org'}:
        doi = unquote(parsed.path).lstrip('/')
        return 'doi:' + doi.casefold() if DOI_RE.fullmatch(doi) else 'url:' + text
    raw_doi = re.sub(r'^doi:\s*', '', text, flags=re.IGNORECASE)
    if DOI_RE.fullmatch(raw_doi):
        return 'doi:' + raw_doi.casefold()
    return 'url:' + text

def candidate_links(row):
    # Upper-case fields are the original record; lower-case fields are outputV2 suggestions.
    return {key for field in ('DOI', 'Url', 'doi', 'url')
            if (key := link_key(row.get(field, '')))}

def read_final_ris(path):
    # Stream only identity fields: long abstracts and notes are irrelevant here.
    record = None
    with path.open('r', encoding='utf-8-sig', errors='replace') as stream:
        for line in stream:
            match = RIS_TAG.match(line.rstrip('\r\n'))
            if not match:
                continue
            tag, value = match.groups()
            tag = tag.upper()
            if tag == 'TY':
                if record is not None:
                    yield record
                record = {'title': '', 'links': set()}
            elif tag == 'ER':
                if record is not None:
                    yield record
                record = None
            elif record is not None:
                if tag in {'TI', 'T1'} and not record['title']:
                    record['title'] = value
                elif tag in {'DO', 'UR', 'L1'}:
                    if key := link_key(value):
                        record['links'].add(key)
    if record is not None:
        yield record

In [3]:
source = pd.read_csv(OUTPUT_V2, dtype=str, keep_default_na=False, encoding='utf-8-sig')
required = {'status', 'Title'}
missing = required - set(source.columns)
if missing:
    raise ValueError(f'outputV2.csv is missing columns: {sorted(missing)}')
needs_review = source.loc[source['status'].str.strip().str.casefold().eq('needs_review')].copy()
if needs_review.empty:
    raise ValueError('No rows with status = needs_review were found in outputV2.csv.')

final_records = list(read_final_ris(FINAL_RIS))
by_title = defaultdict(list)
by_link = defaultdict(list)
for record in final_records:
    if key := title_key(record['title']):
        by_title[key].append(record)
    for key in record['links']:
        by_link[key].append(record)

print(f'outputV2 records: {len(source):,}')
print(f'outputV2 needs_review records: {len(needs_review):,}')
print(f'ISSP final RIS records: {len(final_records):,}')

outputV2 records: 8,211
outputV2 needs_review records: 235
ISSP final RIS records: 12,186


In [ ]:
def compare_row(row):
    title = title_key(row.get('Title', ''))
    links = candidate_links(row)
    same_title = by_title.get(title, []) if title else []
    exact = [record for record in same_title if links & record['links']]
    if exact:
        category = 'exact_link_in_final'
        matches = exact
    elif same_title:
        category = ('same_title_different_link' if any(r['links'] for r in same_title)
                    else 'same_title_no_link')
        matches = same_title
    else:
        other_title = [record for key in links for record in by_link.get(key, [])]
        category = 'link_in_final_but_different_title' if other_title else 'not_in_final'
        matches = other_title
    matched_titles = list(dict.fromkeys(r['title'] for r in matches if r['title']))
    matched_links = sorted({key for r in matches for key in r['links']})
    return pd.Series({
        'Final Comparison': category,
        'Final Matching Titles': ' | '.join(matched_titles),
        'Final Matching Links': ' | '.join(matched_links),
        'Final Matching Record Count': len(matches),
    })

comparison = needs_review.apply(compare_row, axis=1)
result = pd.concat([needs_review, comparison], axis=1)
counts = Counter(result['Final Comparison'])
exact_count = counts['exact_link_in_final']
remaining_count = len(result) - exact_count
assert exact_count + remaining_count == len(needs_review)

print('\nComparison results')
print(f'Original needs_review: {len(needs_review):,}')
print(f'Already has the exact link in final: {exact_count:,}')
print(f'Still needs review: {remaining_count:,}')
for category in ('same_title_different_link', 'same_title_no_link',
                 'link_in_final_but_different_title', 'not_in_final'):
    print(f'  {category}: {counts[category]:,}')

remaining = result.loc[result['Final Comparison'].ne('exact_link_in_final')].copy()
# Never silently overwrite a previously generated review file.
output_path = REVIEW_CSV
if output_path.exists():
    stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    output_path = REVIEW_CSV.with_name(f'{REVIEW_CSV.stem}_{stamp}.csv')
remaining.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f'\nSaved {len(remaining):,} remaining rows to: {output_path}')
display(remaining[['Title', 'url', 'Final Comparison', 'Final Matching Titles']].head(10))


Comparison results
Original needs_review: 235
Already has the exact link in final: 63
Still needs review: 172
  same_title_different_link: 17
  same_title_no_link: 155
  link_in_final_but_different_title: 0
  not_in_final: 0

Saved 172 remaining rows to: D:\UoA\Master\2026 S2\Project\DataSet\needs_review_after_final.csv


,Title,url,Final Comparison,Final Matching Titles
1,"Kön, kontroll och arbetstid: En kvantitativ un...",http://urn.kb.se/resolve?urn=urn:nbn:se:oru:di...,same_title_no_link,"Kön, kontroll och arbetstid: En kvantitativ un..."
4,Matchningen mellan arbetsvärderingar och arbet...,http://urn.kb.se/resolve?urn=urn:nbn:se:umu:di...,same_title_no_link,Matchningen mellan arbetsvärderingar och arbet...
10,Brahmin left versus merchant right. How useful...,https://archive-ouverte.unige.ch/unige:159102,same_title_no_link,Brahmin left versus merchant right. How useful...
29,Marginalisering og velfærdspolitik. Arbejdsløs...,https://www.semanticscholar.org/paper/2c17e28c...,same_title_no_link,Marginalisering og velfærdspolitik. Arbejdsløs...
57,Sosiaalinen tuki sosiaalisen pääoman käytäntön...,https://www.semanticscholar.org/paper/a7b55421...,same_title_no_link,Sosiaalinen tuki sosiaalisen pääoman käytäntön...
173,Social Inequality: International Social Survey...,https://doi.org/10.4232/1.2310,same_title_no_link,Social Inequality: International Social Survey...
293,ISSP 1996 'Role of Government III',http://www.ssoar.info/ssoar/handle/document/19977,same_title_no_link,ISSP 1996 'Role of Government III'
300,Technical report ISSP 2006 Denmark: Role of Go...,https://vbn.aau.dk/ws/files/16328780/2008_2.pdf,same_title_no_link,Technical report ISSP 2006 Denmark: Role of Go...
312,Overview,https://doi.org/10.1016/b978-0-08-021790-1.500...,same_title_no_link,Overview
365,Atheism and Agnosticism in Twenty-First-Centur...,https://pure.qub.ac.uk/en/publications/8e3b99f...,same_title_no_link,Atheism and Agnosticism in Twenty-First-Centur...


: 